📘 **Collab Notebook** семинара можно найти [по ссылке](https://colab.research.google.com/drive/1r1pmlh3Zg-XateE6Lf03rl3JRRyscLxR?usp=sharing).

Домашнее задание будет опубликовано позже.



## 🧠 Домашнее задание 9

---

Домашнее задание необходимо предоставить в формате **ссылки на Google Colab / Jupyter Notebook**
с вашими действиями и ключевыми выводами.

---

📅 **Дедлайны:**

* 🕐 **21 октября 23:59** — мягкий дедлайн
* ⏰ **28 октября 23:59** — жёсткий дедлайн

*До мягкого дедлайна за работу можно получить **10 баллов**, после — **5**.
Работы, отправленные после 28 октября, могут быть проверены преподавателями до конца курса
в формате “зачёт / не зачёт”.*

In [ ]:
!pip install "qdrant-client>=1.7.0" "sentence-transformers>=2.2.2" beir rank-bm25 pytrec_eval rich

In [ ]:
#!pip install sentence-transformers rank_bm25 qdrant-client

In [ ]:
import os
import json
import time
import math
import random
import numpy as np
from collections import defaultdict, Counter
from functools import partial

import torch
from sentence_transformers import SentenceTransformer, CrossEncoder

from beir import util
from beir.datasets.data_loader import GenericDataLoader
from beir.retrieval.evaluation import EvaluateRetrieval

from rank_bm25 import BM25Okapi
from sklearn.feature_extraction.text import TfidfVectorizer

from qdrant_client import QdrantClient
from qdrant_client.models import (
    Distance, VectorParams, PointStruct, Filter, FieldCondition, MatchValue,
    HnswConfigDiff, OptimizersConfigDiff, SearchParams,
    ScalarQuantizationConfig, ScalarType, PayloadSchemaType
)

In [ ]:
import warnings

warnings.filterwarnings("ignore")

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
device

### **1. Выбор датасета — 1 балл**

* Загрузите понравившийся датасет из **BEIR** (но не `scifact` — он уже был на семинаре 🙂).

In [ ]:
data_dir = "./data"
os.makedirs(data_dir, exist_ok=True)

dataset = "scidocs"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

scifact_path = util.download_and_unzip(url, data_dir)
corpus, queries, qrels = GenericDataLoader(scifact_path).load(split="test")

print(f"Docs: {len(corpus)}, Queries: {len(queries)}, qrels: {len(qrels)}")
# Пример документа
sample_id = next(iter(corpus.keys()))
print({ "doc_id": sample_id, "title": corpus[sample_id].get("title"), "text": corpus[sample_id].get("text")[:200] })

In [ ]:
from beir import util, datasets
from beir.datasets.data_loader import GenericDataLoader

# Название датасета
dataset = "fiqa"

# Скачиваем и распаковываем (автоматически кэшируется)
data_path = util.download_and_unzip(
    "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/fiqa.zip",
    "datasets"
)

# Загружаем test-сплит
corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

print(f"Документов: {len(corpus)}")
print(f"Запросов: {len(queries)}")
print(f"qrels: {len(qrels)}")
print(f"Пример документа:\n{list(corpus.values())[0]}")
print(f"Пример запроса:\n{list(queries.values())[0]}")


### Чанкинг документов

### **2. Подготовка датасета — 2 балла**

*Проведите предобработку текстов:*

* Приведение к нижнему регистру и другая нормализация, которую считаете необходимой.
* Токенизация.
* Чанкинг (стратегию чанкирования можно выбрать по желанию).

### Код предобработки

In [ ]:
import re
from nltk.tokenize import word_tokenize
import nltk

nltk.download("punkt", quiet=True)

def preprocess_text(text: str) -> str:
    """Простая очистка текста."""
    text = text.lower()  # нижний регистр
    text = re.sub(r"<.*?>", " ", text)  # HTML
    text = re.sub(r"[^a-z0-9\s]", " ", text)  # только буквы и цифры
    text = re.sub(r"\s+", " ", text).strip()  # убрать лишние пробелы
    return text

def tokenize(text: str):
    return word_tokenize(text)


### Чанкинг документов

In [ ]:
def chunk_text(text, max_words=120, overlap=30):
    words = text.split()
    if len(words) <= max_words:
        return [text]

    chunks = []
    start = 0
    while start < len(words):
        end = min(start + max_words, len(words))
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        if end == len(words):
            break
        start = max(0, end - overlap)
    return chunks

In [ ]:
# Пример применения к первому документу
doc_id, doc = list(corpus.items())[0]
text = f"{doc.get('title', '')} {doc.get('text', '')}"
clean_text = preprocess_text(text)
chunks = chunk_text(clean_text)

print(f"ID: {doc_id}")
print(f"Количество чанков: {len(chunks)}")
print("Первый чанк:", chunks[0][:200])


In [ ]:
from collections import defaultdict
import numpy as np
from tqdm import tqdm

chunk_corpus = {}  # chunk_id -> {"text", "doc_id", "title", "chunk_idx"}
doc_to_chunks = defaultdict(list)

for doc_id, doc in tqdm(corpus.items()):
    raw_text = (doc.get("title", "") + ". " + doc.get("text", "").strip()).strip()
    clean_text = preprocess_text(raw_text)
    chunks = chunk_text(clean_text, max_words=120, overlap=30)

    for idx, ch in enumerate(chunks):
        chunk_id = f"{doc_id}#c{idx}"
        chunk_corpus[chunk_id] = {
            "text": ch,
            "doc_id": doc_id,
            "title": doc.get("title", ""),
            "chunk_idx": idx,
        }
        doc_to_chunks[doc_id].append(chunk_id)

print(f"Total chunks: {len(chunk_corpus)}; avg chunks/doc: {np.mean([len(v) for v in doc_to_chunks.values()]):.2f}")



### **3. Подготовка функции для оценки качества поиска — 1 балл**

* Реализуйте шаблонную функцию для замера качества поиска по основным метрикам:
  **Precision@k**, **Recall@k**, **MAP@k** или **NDCG@k**.

In [ ]:
import numpy as np
from collections import defaultdict

def precision_at_k(relevant_docs, retrieved_docs, k):
    """
    Вычисляет Precision@K для одного запроса.

    Args:
        relevant_docs (set or list): Множество релевантных документов для запроса.
        retrieved_docs (list): Список документов, возвращённых поисковой системой.
        k (int): Глубина топ-K.

    Returns:
        float: Доля релевантных документов среди первых k результатов.
    """
    retrieved_k = retrieved_docs[:k]
    hits = sum(1 for d in retrieved_k if d in relevant_docs)
    return hits / k

def recall_at_k(relevant_docs, retrieved_docs, k):
    """
    Вычисляет Recall@K для одного запроса.

    Args:
        relevant_docs (set or list): Множество релевантных документов для запроса.
        retrieved_docs (list): Список документов, возвращённых поисковой системой.
        k (int): Глубина топ-K.

    Returns:
        float: Доля релевантных документов, найденных в топ-K результатов.
    """
    retrieved_k = retrieved_docs[:k]
    hits = sum(1 for d in retrieved_k if d in relevant_docs)
    return hits / len(relevant_docs) if relevant_docs else 0.0

def average_precision(relevant_docs, retrieved_docs, k):
    """
    Вычисляет Average Precision@K (AP@K) для одного запроса.

    Args:
        relevant_docs (set or list): Множество релевантных документов.
        retrieved_docs (list): Список документов, возвращённых поисковой системой.
        k (int): Глубина топ-K.

    Returns:
        float: Средняя точность по топ-K результатам.
    """
    retrieved_k = retrieved_docs[:k]
    ap = 0.0
    hits = 0
    for i, doc_id in enumerate(retrieved_k):
        if doc_id in relevant_docs:
            hits += 1
            ap += hits / (i + 1)
    return ap / len(relevant_docs) if relevant_docs else 0.0

def dcg(relevant_docs, retrieved_docs, k):
    """
    Вычисляет Discounted Cumulative Gain (DCG@K) для одного запроса.

    Args:
        relevant_docs (dict): Словарь {doc_id: relevance} с релевантностью документов.
        retrieved_docs (list): Список документов, возвращённых поисковой системой.
        k (int): Глубина топ-K.

    Returns:
        float: Значение DCG@K.
    """
    retrieved_k = retrieved_docs[:k]
    dcg_val = 0.0
    for i, doc_id in enumerate(retrieved_k):
        rel = relevant_docs.get(doc_id, 0)
        dcg_val += (2**rel - 1) / np.log2(i + 2)
    return dcg_val

def ndcg_at_k(relevant_docs, retrieved_docs, k):
    """
    Вычисляет Normalized Discounted Cumulative Gain (NDCG@K) для одного запроса.

    Args:
        relevant_docs (dict): Словарь {doc_id: relevance} с релевантностью документов.
        retrieved_docs (list): Список документов, возвращённых поисковой системой.
        k (int): Глубина топ-K.

    Returns:
        float: Значение NDCG@K, нормализованное на идеальный порядок документов.
    """
    ideal_docs = sorted(relevant_docs.values(), reverse=True)[:k]
    idcg = sum((2**rel - 1) / np.log2(i + 2) for i, rel in enumerate(ideal_docs))
    return dcg(relevant_docs, retrieved_docs, k) / idcg if idcg > 0 else 0.0


### Универсальная функция

In [ ]:
def evaluate_retrieval(results, qrels, k_values=[1, 3, 5, 10]):
    """
    Оценивает качество поиска по основным метрикам Information Retrieval для множества запросов.

    Метрики, которые вычисляются:
        - Precision@k
        - Recall@k
        - MAP@k (Mean Average Precision)
        - NDCG@k (Normalized Discounted Cumulative Gain)

    Args:
        results (dict): Результаты поиска, словарь вида:
                        {
                            "query_id_1": [("doc_id_1", score1), ("doc_id_2", score2), ...],
                            "query_id_2": [...],
                            ...
                        }
        qrels (dict): Словарь релевантности документов для каждого запроса:
                      {
                          "query_id_1": set(["doc_id_1", "doc_id_3", ...]),
                          "query_id_2": set([...]),
                          ...
                      }
        k_values (list, optional): Список значений K для вычисления метрик top-K. 
                                   По умолчанию [1, 3, 5, 10].

    Returns:
        dict: Словарь вида:
              {
                  k1: {"precision": float, "recall": float, "map": float, "ndcg": float},
                  k2: {...},
                  ...
              }
              Каждое значение — это среднее значение метрики по всем запросам для данного K.
              
    Example:
        >>> results = {
                "q1": [("d1", 0.9), ("d2", 0.8)],
                "q2": [("d3", 0.7)]
            }
        >>> qrels = {
                "q1": {"d1"},
                "q2": {"d3", "d4"}
            }
        >>> evaluate_retrieval(results, qrels, k_values=[1, 3])
        {
            1: {"precision": 1.0, "recall": 0.75, "map": 0.875, "ndcg": 1.0},
            3: {"precision": 0.5, "recall": 1.0, "map": 0.7083, "ndcg": 0.8333}
        }
    """
    metrics = {k: defaultdict(list) for k in k_values}

    for qid, relevant_docs in qrels.items():
        retrieved_docs = [doc_id for doc_id, _ in results.get(qid, [])]

        for k in k_values:
            metrics[k]["precision"].append(precision_at_k(relevant_docs, retrieved_docs, k))
            metrics[k]["recall"].append(recall_at_k(relevant_docs, retrieved_docs, k))
            metrics[k]["map"].append(average_precision(relevant_docs, retrieved_docs, k))
            metrics[k]["ndcg"].append(ndcg_at_k(relevant_docs, retrieved_docs, k))

    # усреднение по всем запросам
    mean_metrics = {
        k: {m: np.mean(v) for m, v in vals.items()}
        for k, vals in metrics.items()
    }

    return mean_metrics




### **4. Sparse поиск — 2 балла**

* Реализуйте пайплайн разреженного поиска с использованием **BM25** (например, из `rank-bm25`).
* Измерьте результаты метрик для разных `k`.

### Подготовка корпуса

In [ ]:
bm25_corpus = [doc["text"] for doc in chunk_corpus.values()]
bm25_tokenized_corpus = [tokenize(text) for text in bm25_corpus]

bm25 = BM25Okapi(bm25_tokenized_corpus)


### Поиск для всех запросов

In [ ]:
results_bm25 = {}
k = 10  # можно изменить при оценке

for qid, query in tqdm(queries.items()):
    query_tokens = tokenize(preprocess_text(query))
    scores = bm25.get_scores(query_tokens)
    topk_idx = np.argsort(scores)[::-1][:k]
    
    ranked_results = [
        (list(chunk_corpus.keys())[i], scores[i]) for i in topk_idx
    ]
    results_bm25[qid] = ranked_results


#### Оценка качества

In [ ]:
from collections import defaultdict

def aggregate_chunk_results_to_docs(results_chunk, chunk_corpus):
    doc_results = {}
    for qid, chunk_list in results_chunk.items():
        doc_scores = defaultdict(float)
        for chunk_id, score in chunk_list:
            doc_id = chunk_corpus[chunk_id]["doc_id"]
            # можно брать максимум, или среднее, или сумму
            doc_scores[doc_id] = max(doc_scores[doc_id], score)
        # сортируем по убыванию
        sorted_docs = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
        doc_results[qid] = sorted_docs
    return doc_results


In [ ]:
results_bm25_doc = aggregate_chunk_results_to_docs(results_bm25, chunk_corpus)

In [ ]:
bm25_metrics = evaluate_retrieval(results_bm25_doc, qrels, k_values=[1, 3, 5, 10])
for k, vals in bm25_metrics.items():
    print(f"\nK={k}")
    for metric, value in vals.items():
        print(f"  {metric:<10}: {value:.4f}")


### **5. Dense поиск — 2 балла**

* Реализуйте пайплайн плотного поиска с использованием подходящего, на ваш взгляд, эмбеддера.
  Для реализации векторного поиска используйте индекс в предпочитаемой вами векторной БД.
* Измерьте результаты метрик для разных `k`.

### Инициализация Qdrant memory

In [ ]:
from qdrant_client import QdrantClient
from qdrant_client.http.models import VectorParams, Distance

# В памяти (не на сервере)
client = QdrantClient(":memory:")

dim = 384  # размерность эмбеддингов для all-MiniLM-L6-v2
collection="dense_corpus"
# Создаём коллекцию
#client.recreate_collection(
#    collection_name="dense_corpus",
#    vectors_config=VectorParams(size=dim, distance=Distance.COSINE)
#)


In [ ]:
client.recreate_collection(
    collection_name=collection,
    vectors_config={"dense": VectorParams(
            size=dim,
            distance=Distance.COSINE,
            hnsw_config=HnswConfigDiff(
                m=16, # количество двунаправленных связей для каждого узла (по умолчанию 16). Больше связей = более точный поиск, но больше памяти
                ef_construct=100, # размер динамического списка кандидатов при построении индекса, больше = качественнее индекс, но медленнее построение
                full_scan_threshold=500 # при количестве векторов меньше этого порога будет использоваться полный перебор вместо HNSW (быстрее для малых коллекций)
            )
    )},
    optimizers_config=OptimizersConfigDiff(
        indexing_threshold=0,  # немедленная индексация, начнем строить HNSW после этого порога в сегменте
        memmap_threshold=0
    )
)

### Получение эмбеддингов документов

In [ ]:
from sentence_transformers import SentenceTransformer
from tqdm import tqdm

model = SentenceTransformer("all-MiniLM-L6-v2")


def e5_encode_texts(model, texts, is_query=False, batch_size=256):
    """
    Преобразует список текстов в векторные эмбеддинги с помощью модели E5 или совместимой модели SentenceTransformer.

    Для запросов и документов добавляется префикс ("query:" или "passage:") в соответствии с форматом модели,
    что может улучшить качество эмбеддингов в задачах dense retrieval.

    Args:
        model (SentenceTransformer или совместимая модель): Модель для получения эмбеддингов.
        texts (list of str): Список текстов (запросы или документы), которые нужно закодировать.
        is_query (bool, optional): Если True, добавляется префикс "query: " перед текстом, иначе "passage: ".
                                   По умолчанию False.
        batch_size (int, optional): Размер батча для кодирования текстов. По умолчанию 256.

    Returns:
        np.ndarray: Массив эмбеддингов формы (len(texts), embedding_dim) с нормализованными векторами.

    Example:
        >>> queries = ["What is machine learning?", "Define deep learning"]
        >>> embeddings = e5_encode_texts(model, queries, is_query=True)
        >>> embeddings.shape
        (2, 768)
    """
    prefix = "query: " if is_query else "passage: "
    it = []
    for t in texts:
        t = t.strip()
        it.append(prefix + t)
    vecs = model.encode(
        it,
        batch_size=batch_size,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=True
    )
    return vecs


# Создаём список текстов чанков
#texts = [chunk["text"] for chunk in chunk_corpus.values()]
#chunk_ids = list(chunk_corpus.keys())

# Генерация эмбеддингов
#embeddings = model.encode(texts, batch_size=64, show_progress_bar=True)


In [ ]:
chunk_ids = list(chunk_corpus.keys())
chunk_texts = [chunk_corpus[cid]["text"] for cid in chunk_ids]
chunk_embeddings = e5_encode_texts(model, chunk_texts, is_query=False, batch_size=256)
dim = chunk_embeddings.shape[1]
print(f"E5 dense dimension: {dim}")

#### Индексирование в Qdrant

In [ ]:
BATCH = 2048
points = []

for i, cid in enumerate(chunk_ids):
    vec = chunk_embeddings[i].astype(np.float32)
    payload = {
        "chunk_id": cid,
        "doc_id": chunk_corpus[cid]["doc_id"],
        "title": chunk_corpus[cid]["title"],
        "chunk_idx": chunk_corpus[cid]["chunk_idx"]
    }
    pt = PointStruct(
        id=i, vector={"dense": vec}, payload=payload
    )
    points.append(pt)
    if len(points) == BATCH or i == len(chunk_ids) - 1:
        client.upsert(collection_name=collection, points=points)
        points = []

In [ ]:
def search_dense_qdrant(client, query, collection_name="dense_corpus", top_k=50, hnsw_ef=16):
    """
    Выполняет плотный (dense) поиск по коллекции Qdrant для одного запроса.

    Функция использует эмбеддинги (например, E5) для преобразования запроса в вектор
    и ищет наиболее похожие векторные представления документов в Qdrant. 
    Результаты агрегируются по уровню документов (doc_id).

    Args:
        client (QdrantClient): Экземпляр клиента Qdrant.
        query (str): Текст запроса.
        collection_name (str, optional): Имя коллекции в Qdrant. По умолчанию "dense_corpus".
        top_k (int, optional): Количество возвращаемых ближайших соседей. По умолчанию 50.
        hnsw_ef (int, optional): Параметр HNSW индекса для управления качеством поиска. 
                                 По умолчанию 16.

    Returns:
        tuple:
            dict: Словарь {doc_id: score} с агрегированными максимальными оценками по документам.
            list: Список сырых результатов поиска по чанкам, каждый элемент содержит payload и score.
    
    Example:
        >>> doc_scores, raw_hits = search_dense_qdrant(client, "What is machine learning?", top_k=10)
        >>> doc_scores
        {'doc_1': 0.87, 'doc_5': 0.81, ...}
        >>> raw_hits[0].payload
        {'chunk_id': 'doc_1#c0', 'doc_id': 'doc_1', 'title': 'Intro to ML'}
    """
    qv = e5_encode_texts(model, [query], is_query=True, batch_size=1)[0].astype(np.float32)
    res = client.search(
        collection_name=collection_name,
        query_vector=("dense", qv),
        limit=top_k,
        with_payload=True,
        search_params=SearchParams(hnsw_ef=hnsw_ef)
    )

    # агрегируем по doc_id
    doc_scores = defaultdict(float)
    for r in res:
        did = r.payload["doc_id"]
        doc_scores[did] = max(doc_scores[did], r.score)
    return doc_scores, res  # doc-level scores + raw chunk hits


In [ ]:
results_dense_doc = {}
results_dense_chunk = {}

for qid, query in queries.items():
    doc_scores, chunk_hits = search_dense_qdrant(client, query, collection_name="dense_corpus", top_k=10)
    results_dense_doc[qid] = sorted(doc_scores.items(), key=lambda x: x[1], reverse=True)
    results_dense_chunk[qid] = chunk_hits

In [ ]:
# Вычисляем метрики
k_values = [1, 3, 5, 10]
dense_metrics = evaluate_retrieval(results_dense_doc, qrels, k_values=k_values)

for k, vals in dense_metrics.items():
    print(f"\nK={k}")
    for metric, value in vals.items():
        print(f"  {metric:<10}: {value:.4f}")


### **6. Hybrid поиск — 2 балла**

* Реализуйте пайплайн гибридного поиска, объединив **dense** и **sparse** поиски.
  Постарайтесь подобрать параметры и оптимальную стратегию объединения результатов.
* Измерьте результаты метрик для разных `k`.


#### 🧩 Концепция гибридного поиска

Цель — объединить:

* **разреженные оценки** (BM25 → tf-idf-подобная релевантность),
* **плотные оценки** (Dense → косинусное сходство или dot-product).

Чтобы это корректно работало, нам нужно:

1. **Нормализовать** оценки (например, min-max или z-score).
2. **Взвешенно объединить** их с помощью коэффициента `α`:
   $$
   score_{hybrid} = α * score_{dense} + (1 - α) * score_{bm25}
   $$
3. **Отсортировать** результаты и измерить метрики.



In [ ]:
import numpy as np
from collections import defaultdict
from tqdm import tqdm

# --- 1. Sparse часть (BM25 по документам) ---
doc_ids = list(corpus.keys())
doc_texts = [(corpus[d].get("title","") + " " + corpus[d].get("text","")).strip() for d in doc_ids]
tokenized = [t.lower().split() for t in doc_texts]
bm25 = BM25Okapi(tokenized)

def bm25_doc_scores(query, top_k=100):
    scores = bm25.get_scores(query.lower().split())
    top_idx = np.argsort(scores)[::-1][:top_k]
    return {doc_ids[i]: float(scores[i]) for i in top_idx}


# --- 2. Dense часть (через Qdrant) ---
#def search_dense_qdrant(query, top_k=100, hnsw_ef=64):
#    qv = e5_encode_texts([query], is_query=True, batch_size=1)[0].astype(np.float32)
#    res = client.search(
#        collection_name="dense_corpus",
#        query_vector=("dense", qv),
#        limit=top_k,
#        with_payload=True,
#        search_params=SearchParams(hnsw_ef=hnsw_ef)
#    )
#    # агрегируем chunk-результаты до doc-уровня
#    doc_scores = defaultdict(float)
#    for r in res:
#        did = r.payload["doc_id"]
#        doc_scores[did] = max(doc_scores[did], r.score)
#    return doc_scores


# --- 3. Гибридный скор ---
def hybrid_scores(client, query, alpha=0.5, top_k=100, hnsw_ef=64):
    dscores, _ = search_dense_qdrant(client, query, top_k=top_k, hnsw_ef=hnsw_ef)
    bscores = bm25_doc_scores(query, top_k=top_k)
    all_docs = set(dscores.keys()) | set(bscores.keys())

    # z-нормализация (по топ-k)
    def z_norm(d):
        if not d:
            return {}
        vals = np.array(list(d.values()))
        mu, std = vals.mean(), (vals.std() + 1e-6)
        return {k: (v - mu) / std for k, v in d.items()}

    dz, bz = z_norm(dscores), z_norm(bscores)

    fused = {}
    for d in all_docs:
        fused[d] = alpha * dz.get(d, -5.0) + (1 - alpha) * bz.get(d, -5.0)

    fused_top = dict(sorted(fused.items(), key=lambda x: x[1], reverse=True)[:max(1000, top_k)])
    return fused_top


# --- 4. Пакетная оценка для всех запросов ---
def evaluate_hybrid(alpha=0.5, top_k=10, k_values=[1, 3, 5, 10]):
    results_hybrid = {}
    
    for qid, query in tqdm(queries.items(), desc=f"Hybrid search α={alpha}"):
        fused_scores = hybrid_scores(client, query, alpha=alpha, top_k=top_k)
        sorted_docs = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
        results_hybrid[qid] = sorted_docs[:top_k]

    hybrid_metrics = evaluate_retrieval(results_hybrid, qrels, k_values=k_values)

    print(f"\nHybrid search (α={alpha})")
    for k, vals in hybrid_metrics.items():
        print(f"\nK={k}")
        for metric, value in vals.items():
            print(f"  {metric:<10}: {value:.4f}")

    return hybrid_metrics


In [ ]:
# Проверим разные альфы
#for a in [0.3, 0.5, 0.7]:
#    evaluate_hybrid(alpha=a, top_k=100)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm


def evaluate_hybrid_grid(client, alpha_list=(0.1, 0.3, 0.5, 0.7, 0.9), top_k=100, k_values=[1, 3, 5, 10]):
    """
    Прогоняет гибридный поиск для разных значений alpha и возвращает статистику.
    Возвращает:
        results_summary: list[dict] — метрики для каждого alpha
    """
    results_summary = []

    for a in alpha_list:
        results_hybrid = {}
        for qid, query in tqdm(queries.items(), desc=f"Hybrid search α={a}"):
            fused_scores = hybrid_scores(client, query, alpha=a, top_k=top_k)
            sorted_docs = sorted(fused_scores.items(), key=lambda x: x[1], reverse=True)
            results_hybrid[qid] = sorted_docs[:top_k]

        hybrid_metrics = evaluate_retrieval(results_hybrid, qrels, k_values=k_values)
        ndcg10 = hybrid_metrics[10]["ndcg"]
        map10 = hybrid_metrics[10]["map"]

        results_summary.append({
            "alpha": a,
            "metrics": hybrid_metrics,
            "ndcg@10": ndcg10,
            "map@10": map10,
        })

        # печать метрик в стиле как раньше
        print(f"\nHybrid search (α={a})")
        for k, vals in hybrid_metrics.items():
            print(f"\nK={k}")
            for metric, value in vals.items():
                print(f"  {metric:<10}: {value:.4f}")

    return results_summary


def plot_hybrid_metrics(results_summary):
    """
    Строит графики зависимости метрик от alpha.
    """
    alphas = [r["alpha"] for r in results_summary]
    ndcg_vals = [r["ndcg@10"] for r in results_summary]
    map_vals = [r["map@10"] for r in results_summary]

    plt.figure(figsize=(8, 5))
    plt.plot(alphas, ndcg_vals, marker="o", label="NDCG@10")
    plt.plot(alphas, map_vals, marker="s", label="MAP@10")
    plt.xlabel("α (вес Dense компоненты)")
    plt.ylabel("Метрика качества")
    plt.title("Зависимость метрик от α в гибридном поиске")
    plt.legend()
    plt.grid(True)
    plt.show()

    # выводим лучшую α
    best = max(results_summary, key=lambda x: x["ndcg@10"])
    print(f"\n🔥 Лучшая α по NDCG@10 = {best['alpha']:.2f}, "
          f"NDCG@10 = {best['ndcg@10']:.4f}, MAP@10 = {best['map@10']:.4f}")


In [ ]:
results_summary = evaluate_hybrid_grid(
    client=client,
    alpha_list=np.linspace(0.1, 0.9, 5),
    top_k=100
)

plot_hybrid_metrics(results_summary)


In [ ]:
import pandas as pd
df_results = pd.DataFrame(results_summary)
df_results

In [ ]:
# 2. Сохраняем в CSV
df_results.to_csv("hybrid_search_metrics.csv", index=False)

In [ ]:
import matplotlib.pyplot as plt

def plot_all_metrics_from_df(df_results):
    """
    Строит графики всех метрик (precision, recall, map, ndcg) для всех K по α,
    когда колонка metrics уже содержит словари.
    """
    metrics = ["precision", "recall", "map", "ndcg"]
    
    # Берем все K из словаря metrics первой строки
    k_values = sorted(df_results['metrics'].iloc[0].keys())

    plt.figure(figsize=(15, 10))

    for i, metric in enumerate(metrics, 1):
        plt.subplot(2, 2, i)
        for k in k_values:
            vals = df_results['metrics'].apply(lambda m: m[k][metric])
            plt.plot(df_results['alpha'], vals, marker='o', label=f"K={k}")
        plt.xlabel("α (вес Dense компоненты)")
        plt.ylabel(metric.upper())
        plt.title(f"{metric.upper()} vs α")
        plt.grid(True)
        plt.legend()

    plt.tight_layout()
    plt.show()

    # выводим лучшую alpha по NDCG@10
    best_row = df_results.sort_values('ndcg@10', ascending=False).iloc[0]
    print(f"\n🔥 Лучшая α по NDCG@10 = {best_row['alpha']:.2f}, "
          f"NDCG@10 = {best_row['ndcg@10']:.4f}, MAP@10 = {best_row['map@10']:.4f}")


In [ ]:
plot_all_metrics_from_df(df_results)



## 1️⃣ Влияние веса Dense (`α`) на метрики

| α   | ndcg@10 | map@10 |
| --- | ------- | ------ |
| 0.1 | 0.184   | 0.137  |
| 0.3 | 0.240   | 0.186  |
| 0.5 | 0.297   | 0.227  |
| 0.7 | 0.349   | 0.275  |
| 0.9 | 0.367   | 0.294  |

**Выводы:**

* Чем больше вес Dense (α ближе к 1), тем **лучше метрики NDCG@10 и MAP@10**.
* Sparse компонент BM25 полезен при малых α, но для этого датасета он явно слабее dense эмбеддингов (E5/косинусная близость).
* Оптимальное α по NDCG@10 ≈ **0.9** — т.е. плотный поиск важнее, но BM25 всё же даёт небольшой бонус.

---

## 2️⃣ Поведение по K

* **Precision падает** с ростом K, что естественно: чем больше топ-K, тем больше «не релевантных» документов.
* **Recall растет** с ростом K — мы видим больше релевантных документов.
* **NDCG и MAP** растут умеренно с увеличением α, особенно при K=10, что указывает на то, что Dense поиск корректно ранжирует релевантные документы в верхней части.

---

## 3️⃣ Практические рекомендации

1. **Для этого датасета** α≈0.9 даёт оптимальный баланс — гибрид почти полностью полагается на Dense поиск.
2. **BM25 полезен** для небольших α или когда dense-эмбеддинги не доступны.
3. Если цель — **top-1 результат**, стоит проверять α отдельно, так как влияние sparse может быть сильнее для «редких» запросов.
4. Можно использовать **RRF или другие методы ранжирования** для дополнительного улучшения результатов, особенно если Sparse сильнее для определённых типов запросов.
